# 07 - LLM Usage Record

## Purpose
Builds `AUDIT.LLM_USAGE` with token counts from `AI_COUNT_TOKENS`.

## What this notebook does
For each `AI_COMPLETE` stage in the pipeline, the notebook:
1. Rebuilds the prompt exactly as the pipeline built it, from the page text and the active prompts and schemas in
   `PIPELINE_CONFIG` and `DOC_TYPE_CONFIG`.
2. Takes the model output that is already stored in the pipeline tables.
3. Counts tokens on both with `AI_COUNT_TOKENS`, using the model configured for that step.

No `AI_COMPLETE` calls are made, so the build has no model cost.

| Step | Model (from `PIPELINE_CONFIG`) | Rows written |
|---|---|---|
| `LANGUAGE_CHECK` pass 1 | `lang_check_haiku_model` | One per non-empty page |
| `LANGUAGE_CHECK` pass 2 | `lang_check_sonnet_model` | One per page that went to the second pass |
| `TRANSLATE` | `translate_model` | One per page translated |
| `CLASSIFY` | `classify_model` | One per classified file |
| `EXTRACTION` | `extract_model` | One per parent file extracted |

`AI_PARSE_DOCUMENT` is not in this table. It is billed per page, so its cost comes from the page count.

### Where the counted text comes from

| Step | Input | Output |
|---|---|---|
| Language pass 1 and pass 2 | Exact prompt rebuilt from config and page text | Rebuilt from `LANGUAGE_RESULT` and the stored reason (approximate, about 20 tokens per page) |
| Translation | Exact | Stored `PAGE_CONTENT_TRANSLATED` (very close) |
| Classification | Exact | Rebuilt from the `DOCUMENTS_CLASSIFIED` columns (approximate) |
| Extraction | Exact | Stored `EXTRACTED_JSON`, re-serialized as compact JSON (close, but not the raw response text) |

Only calls that have a stored output are counted. A page or file that never reached a step is left out.

## Outputs
| Table | What is written |
|---|---|
| `AUDIT.LLM_USAGE` | One row per call with `DOC_ID`, `PIPELINE_STEP`, `MODEL_NAME`, `TOKENS_IN`, `TOKENS_OUT` and `CALLED_AT` set to the time of the rebuild. `CHILD_DOC_ID` is NULL |

In [ ]:
import json
import pandas as pd
from datetime import datetime
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
PROCESSING_SCHEMA = 'PROCESSING'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA      = 'AUDIT'

PAGES      = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES"
CLASSIFIED = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED"
EXTRACTED  = f"{DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED"
LLM_USAGE  = f"{DB}.{AUDIT_SCHEMA}.LLM_USAGE"

s = get_active_session()

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")

CONFIG_KEYS = [
    'lang_check_haiku_model',  'lang_check_haiku_prompt',
    'lang_check_sonnet_model', 'lang_check_sonnet_prompt',
    'translate_model',         'translate_prompt',
    'classify_model',          'classify_prompt',
    'extract_model', 'extract_prompt'
]
key_list = ', '.join(f"'{k}'" for k in CONFIG_KEYS)

config = {
    row['CONFIG_KEY']: row['CONFIG_VALUE']
    for row in s.sql(f"""
        SELECT CONFIG_KEY, CONFIG_VALUE
        FROM {DB}.{CONFIG_SCHEMA}.PIPELINE_CONFIG
        WHERE CONFIG_KEY IN ({key_list}) AND IS_ACTIVE = TRUE
    """).collect()
}
missing = [k for k in CONFIG_KEYS if not config.get(k)]
if missing:
    raise ValueError(f"Missing config keys in PIPELINE_CONFIG: {missing}")

HAIKU_MODEL      = config['lang_check_haiku_model']
HAIKU_PROMPT     = config['lang_check_haiku_prompt']
SONNET_MODEL     = config['lang_check_sonnet_model']
SONNET_PROMPT    = config['lang_check_sonnet_prompt']
TRANSLATE_MODEL  = config['translate_model']
TRANSLATE_PROMPT = config['translate_prompt']
CLASSIFY_MODEL   = config['classify_model']
CLASSIFY_PROMPT  = config['classify_prompt']
EXTRACT_MODEL    = config['extract_model']
PACKAGE_PROMPT_TEMPLATE = config['extract_prompt']


In [ ]:
def build_fields_block(mandatory_fields, optional_fields):
    lines = ["MANDATORY FIELDS (return null if not found - never guess):"]
    for f in mandatory_fields:
        lines.append(f"\n  Field: {f['field_id']}")
        lines.append(f"  {f['question']}")
    if optional_fields:
        lines.append("\nOPTIONAL FIELDS (return null if not found):")
        for f in optional_fields:
            lines.append(f"\n  Field: {f['field_id']}")
            lines.append(f"  {f['question']}")
    return '\n'.join(lines)

In [ ]:
usage_frames = []

def count_step(step, pipeline_step, model, base_sql, params=None):
    # base_sql returns DOC_ID, PROMPT (the text sent to the model) and
    # STORED_RESPONSE (the output already in the pipeline tables).
    scratch = f"{DB}.{AUDIT_SCHEMA}.USAGE_SCRATCH_{step}"
    s.sql(f"""
        CREATE OR REPLACE TEMPORARY TABLE {scratch} AS
        SELECT DOC_ID, PROMPT, STORED_RESPONSE AS RESPONSE
        FROM ({base_sql})
        WHERE STORED_RESPONSE IS NOT NULL
    """, params=params).collect()

    df = s.sql(f"""
        SELECT DOC_ID,
               AI_COUNT_TOKENS('ai_complete', '{model}', PROMPT)   AS TOKENS_IN,
               AI_COUNT_TOKENS('ai_complete', '{model}', RESPONSE) AS TOKENS_OUT
        FROM {scratch}
    """).to_pandas()

    df['STEP']          = step
    df['PIPELINE_STEP'] = pipeline_step
    df['MODEL_NAME']    = model
    usage_frames.append(df)
    info(f"[{step}] {len(df)} row(s) | {int(df['TOKENS_IN'].sum()):,} in | {int(df['TOKENS_OUT'].sum()):,} out")

In [ ]:
# Language pass 1
count_step('LANGUAGE_PASS1', 'LANGUAGE_CHECK', HAIKU_MODEL, f"""
    SELECT p.DOC_ID,
           REPLACE(?, '{{page_text}}', p.PAGE_CONTENT) AS PROMPT,
           CASE WHEN p.LANGUAGE_RESULT IS NULL THEN NULL
                ELSE '{{"label": "' || IFF(p.LANGUAGE_RESULT = 'ENGLISH', 'ENGLISH', 'NON_ENGLISH') ||
                     '", "reason": "' ||
                     REGEXP_REPLACE(COALESCE(p.LANGUAGE_CLASSIFICATION_REASON, ''), '^(Haiku|Sonnet): ', '') ||
                     '"}}' END AS STORED_RESPONSE
    FROM {PAGES} p
    WHERE p.PAGE_CONTENT IS NOT NULL AND TRIM(p.PAGE_CONTENT) <> ''
""", params=[HAIKU_PROMPT])

# Language pass 2 (pages that went on to the second pass)
count_step('LANGUAGE_PASS2', 'LANGUAGE_CHECK', SONNET_MODEL, f"""
    SELECT p.DOC_ID,
           REPLACE(?, '{{page_text}}', p.PAGE_CONTENT) AS PROMPT,
           '{{"label": "' || p.LANGUAGE_RESULT || '", "reason": "' ||
           REGEXP_REPLACE(p.LANGUAGE_CLASSIFICATION_REASON, '^Sonnet: ', '') || '"}}' AS STORED_RESPONSE
    FROM {PAGES} p
    WHERE p.LANGUAGE_CLASSIFICATION_REASON LIKE 'Sonnet:%'
      AND p.PAGE_CONTENT IS NOT NULL AND TRIM(p.PAGE_CONTENT) <> ''
""", params=[SONNET_PROMPT])

# Translation
count_step('TRANSLATE', 'TRANSLATE', TRANSLATE_MODEL, f"""
    SELECT p.DOC_ID,
           REPLACE(?, '{{page_text}}', p.PAGE_CONTENT) AS PROMPT,
           p.PAGE_CONTENT_TRANSLATED AS STORED_RESPONSE
    FROM {PAGES} p
    WHERE p.LANGUAGE_RESULT = 'NON_ENGLISH'
      AND p.PAGE_CONTENT IS NOT NULL AND TRIM(p.PAGE_CONTENT) <> ''
""", params=[TRANSLATE_PROMPT])

# Classification
prompt_prefix = CLASSIFY_PROMPT.split('{pages}')[0]
count_step('CLASSIFY', 'CLASSIFY', CLASSIFY_MODEL, f"""
    WITH stored AS (
        SELECT DOC_ID,
               TO_JSON(OBJECT_CONSTRUCT('segments', ARRAY_AGG(OBJECT_CONSTRUCT(
                   'doc_type', DOC_TYPE, 'page_start', PAGE_START, 'page_end', PAGE_END,
                   'confidence', CONFIDENCE, 'boundary_signal', BOUNDARY_SIGNAL,
                   'document_description', DOCUMENT_DESCRIPTION)))) AS STORED_RESPONSE
        FROM {CLASSIFIED}
        GROUP BY DOC_ID
    )
    SELECT p.DOC_ID,
           ? || LISTAGG(
                   '[PAGE ' || p.PAGE_NUMBER || ']' || CHR(10) ||
                   COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
                   '\\n\\n'
                ) WITHIN GROUP (ORDER BY p.PAGE_NUMBER) AS PROMPT,
           MAX(st.STORED_RESPONSE) AS STORED_RESPONSE
    FROM {PAGES} p
    JOIN stored st ON p.DOC_ID = st.DOC_ID
    WHERE p.PAGE_CONTENT IS NOT NULL
    GROUP BY p.DOC_ID
""", params=[prompt_prefix])

# Extraction
doc_types = [r['DOC_TYPE'] for r in s.sql(f"""
    SELECT DISTINCT DOC_TYPE FROM {CLASSIFIED}
    WHERE DOC_TYPE IS NOT NULL AND DOC_TYPE != 'unknown'
""").collect()]

field_block_rows = []
for doc_type in doc_types:
    cfg_rows = s.sql(f"""
        SELECT MANDATORY_FIELDS, OPTIONAL_FIELDS
        FROM {DB}.{CONFIG_SCHEMA}.DOC_TYPE_CONFIG
        WHERE DOC_TYPE = '{doc_type}' AND IS_ACTIVE = TRUE
    """).collect()
    if not cfg_rows:
        warning(f"No active config for '{doc_type}' - skipped")
        continue
    mandatory = json.loads(cfg_rows[0]['MANDATORY_FIELDS'] or '[]')
    optional  = json.loads(cfg_rows[0]['OPTIONAL_FIELDS']  or '[]')
    if mandatory:
        field_block_rows.append({'DOC_TYPE': doc_type,
                                 'FIELDS_BLOCK': build_fields_block(mandatory, optional)})

s.write_pandas(pd.DataFrame(field_block_rows), table_name='USAGE_FIELD_BLOCKS',
               database=DB, schema=AUDIT_SCHEMA,
               overwrite=True, auto_create_table=True, table_type='temporary')
FIELD_BLOCKS = f"{DB}.{AUDIT_SCHEMA}.USAGE_FIELD_BLOCKS"

count_step('EXTRACTION', 'EXTRACTION', EXTRACT_MODEL, f"""
    WITH segs AS (
        SELECT c.DOC_ID, c.CHILD_DOC_ID, c.DOC_TYPE, c.PAGE_START, c.PAGE_END, fb.FIELDS_BLOCK,
               'S' || ROW_NUMBER() OVER (PARTITION BY c.DOC_ID
                                         ORDER BY c.PAGE_START, c.CHILD_DOC_ID) AS SEG_LABEL
        FROM {CLASSIFIED} c
        JOIN {FIELD_BLOCKS} fb ON c.DOC_TYPE = fb.DOC_TYPE
    ),
    seg_block AS (
        SELECT DOC_ID,
               LISTAGG('=== SEGMENT ' || SEG_LABEL || ' | ' || DOC_TYPE ||
                       ' | pages ' || PAGE_START || '-' || PAGE_END || ' ===' ||
                       CHR(10) || FIELDS_BLOCK,
                       '\\n\\n') WITHIN GROUP (ORDER BY PAGE_START) AS SEGMENTS_BLOCK
        FROM segs
        GROUP BY DOC_ID
    ),
    stored AS (
        SELECT sg.DOC_ID,
               TO_JSON(OBJECT_AGG(sg.SEG_LABEL,
                       COALESCE(TRY_PARSE_JSON(e.EXTRACTED_JSON::VARCHAR), e.EXTRACTED_JSON))) AS STORED_RESPONSE
        FROM segs sg
        JOIN {EXTRACTED} e ON e.CHILD_DOC_ID = sg.CHILD_DOC_ID
        GROUP BY sg.DOC_ID
    ),
    page_tags AS (
        SELECT p.DOC_ID, p.PAGE_NUMBER,
               COALESCE(MAX(sg.SEG_LABEL || ' ' || sg.DOC_TYPE),
                        'not extracted: ' || MAX(ac.DOC_TYPE),
                        'unassigned') AS PAGE_TAG
        FROM {PAGES} p
        LEFT JOIN segs sg
            ON  p.DOC_ID = sg.DOC_ID
            AND p.PAGE_NUMBER BETWEEN sg.PAGE_START AND sg.PAGE_END
        LEFT JOIN {CLASSIFIED} ac
            ON  p.DOC_ID = ac.DOC_ID
            AND p.PAGE_NUMBER BETWEEN ac.PAGE_START AND ac.PAGE_END
        WHERE p.DOC_ID IN (SELECT DOC_ID FROM seg_block)
        GROUP BY p.DOC_ID, p.PAGE_NUMBER
    ),
    page_block AS (
        SELECT p.DOC_ID,
               LISTAGG('[PAGE ' || p.PAGE_NUMBER || ' | ' || t.PAGE_TAG || ']' || CHR(10) ||
                       COALESCE(p.PAGE_CONTENT_TRANSLATED, p.PAGE_CONTENT),
                       '\\n\\n') WITHIN GROUP (ORDER BY p.PAGE_NUMBER) AS PAGE_TEXT
        FROM {PAGES} p
        JOIN page_tags t
            ON  p.DOC_ID      = t.DOC_ID
            AND p.PAGE_NUMBER = t.PAGE_NUMBER
        WHERE p.PAGE_CONTENT IS NOT NULL
        GROUP BY p.DOC_ID
    )
    SELECT sb.DOC_ID,
           REPLACE(REPLACE(?, '{{segments_block}}', sb.SEGMENTS_BLOCK),
                   '{{page_text}}', pb.PAGE_TEXT) AS PROMPT,
           st.STORED_RESPONSE
    FROM seg_block sb
    JOIN page_block pb ON sb.DOC_ID = pb.DOC_ID
    LEFT JOIN stored st ON sb.DOC_ID = st.DOC_ID
""", params=[PACKAGE_PROMPT_TEMPLATE])

In [ ]:
usage = pd.concat(usage_frames, ignore_index=True)
summary = (usage.groupby(['STEP', 'PIPELINE_STEP', 'MODEL_NAME'])
                .agg(ROWS=('DOC_ID', 'size'), FILES=('DOC_ID', 'nunique'),
                     TOKENS_IN=('TOKENS_IN', 'sum'), TOKENS_OUT=('TOKENS_OUT', 'sum'))
                .reset_index())
print(summary.to_string(index=False))
print(f"\nTotal rows: {len(usage)} | tokens in: {int(usage['TOKENS_IN'].sum()):,} | "
      f"tokens out: {int(usage['TOKENS_OUT'].sum()):,}")

In [ ]:
# Backup (zero-copy clone)
backup = f"{LLM_USAGE}_BACKUP_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
s.sql(f"CREATE TABLE {backup} CLONE {LLM_USAGE}").collect()
info(f"Backed up LLM_USAGE to {backup}")

# Stage the counted rows (temporary table, created before the transaction starts)
staging = usage[['DOC_ID', 'PIPELINE_STEP', 'MODEL_NAME', 'TOKENS_IN', 'TOKENS_OUT']].copy()
staging['TOKENS_IN']  = staging['TOKENS_IN'].astype(int)
staging['TOKENS_OUT'] = staging['TOKENS_OUT'].astype(int)
s.write_pandas(staging, table_name='LLM_USAGE_REBUILD_STAGING', database=DB, schema=AUDIT_SCHEMA,
               overwrite=True, auto_create_table=True, table_type='temporary')

# Replace the rows 
s.sql("BEGIN").collect()
try:
    s.sql(f"DELETE FROM {LLM_USAGE}").collect()
    s.sql(f"""
        INSERT INTO {LLM_USAGE}
            (DOC_ID, CHILD_DOC_ID, PIPELINE_STEP, MODEL_NAME, TOKENS_IN, TOKENS_OUT, CALLED_AT)
        SELECT DOC_ID, NULL, PIPELINE_STEP, MODEL_NAME, TOKENS_IN, TOKENS_OUT, CURRENT_TIMESTAMP()
        FROM {DB}.{AUDIT_SCHEMA}.LLM_USAGE_REBUILD_STAGING
    """).collect()
    s.sql("COMMIT").collect()
except Exception:
    s.sql("ROLLBACK").collect()
    raise
info(f"LLM_USAGE replaced with {len(staging)} row(s)")

# Drop temp tables
for step in ['LANGUAGE_PASS1', 'LANGUAGE_PASS2', 'TRANSLATE', 'CLASSIFY', 'EXTRACTION']:
    s.sql(f"DROP TABLE IF EXISTS {DB}.{AUDIT_SCHEMA}.USAGE_SCRATCH_{step}").collect()
s.sql(f"DROP TABLE IF EXISTS {DB}.{AUDIT_SCHEMA}.USAGE_FIELD_BLOCKS").collect()

s.sql(f"""
    SELECT PIPELINE_STEP, MODEL_NAME, COUNT(*) AS ROWS_, SUM(TOKENS_IN) AS TOKENS_IN, SUM(TOKENS_OUT) AS TOKENS_OUT
    FROM {LLM_USAGE} GROUP BY 1, 2 ORDER BY 1, 2
""").show()